# Lab 10: Azure AI Search Grounding — Enterprise Knowledge Base
---
**SmartClaims AI Agent** | Microsoft Foundry Agent Service (v2.x)

## Objective
Create a search index, populate it with policy documents, and connect your agent to it for production-grade RAG. You'll learn:
- The difference between managed FileSearch (Lab 3) and Azure AI Search
- Creating and populating a search index programmatically
- Configuring **AzureAISearchTool** with Foundry connections
- Enterprise search features: hybrid search, security trimming, custom analyzers

## Lab 3 vs Lab 10
| Feature | Lab 3 (FileSearch) | Lab 10 (AI Search) |
|---------|-------------------|-------------------|
| Scale | Small (< 100 files) | Large (millions of docs) |
| Search | Vector only | Hybrid (vector + keyword + semantic) |
| Index | Managed by platform | You control the index |
| Security | No trimming | Per-doc ACLs supported |
| Setup | Just upload files | Index + connection needed |
| Best for | Prototyping | Production / Enterprise |

> **Prerequisites:**
> - Azure AI Search resource connected to your Foundry project
> - `.env` variables: `AI_SEARCH_CONNECTION_NAME`, `AI_SEARCH_INDEX_NAME`
> - Run Lab 2 first (generates `contoso_insurance_policy.md`)

## Step 1: Architecture Comparison

```
Lab 3 — FileSearchTool (Managed Vector Store):
┌───────────────────────────────────────────────┐
│  Upload files → Platform chunks & embeds      │
│  → Managed vector store → Agent queries it    │
│  Great for: quick prototyping, small doc sets  │
└───────────────────────────────────────────────┘

Lab 10 — AzureAISearchTool (Your Index):
┌───────────────────────────────────────────────┐
│  Create index → Chunk & upload documents       │
│  → Agent connects via Foundry connection       │
│  → Hybrid search (vector + keyword + semantic) │
│  Great for: production, large doc sets,        │
│  security trimming, custom analyzers            │
└───────────────────────────────────────────────┘
```

## Step 2: Import Dependencies

In [ ]:
%pip install azure-search-documents --quiet

In [ ]:
import sys, os, re, hashlib
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.config import get_clients, MODEL, POLICY_DOC
from azure.ai.projects.models import (
    PromptAgentDefinition,
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
)

project_client, openai_client = get_clients()
print("✅ Clients created")

## Step 3: Resolve Search Connection from Foundry

Instead of hardcoding search keys, we retrieve the endpoint and API key directly from the Foundry project connection. This is the production-recommended pattern.

In [ ]:
connection_name = os.environ.get("AI_SEARCH_CONNECTION_NAME", "")
index_name = os.environ.get("AI_SEARCH_INDEX_NAME", "contoso-insurance-index")

if not connection_name:
    raise ValueError(
        "AI_SEARCH_CONNECTION_NAME not set in .env\n"
        "Set: AI_SEARCH_CONNECTION_NAME=<your-connection-name>"
    )

# Resolve endpoint from Foundry connection
azs_connection = project_client.connections.get(connection_name)
connection_id = azs_connection.id
search_endpoint = azs_connection.target

# Get API key from the connection (with secrets)
azs_with_secrets = project_client.connections.get(
    connection_name, include_credentials=True
)
api_key = azs_with_secrets.credentials.api_key
credential = AzureKeyCredential(api_key)

print(f"✅ Connection:  {connection_name}")
print(f"✅ Endpoint:    {search_endpoint}")
print(f"✅ API Key:     Retrieved from Foundry (not hardcoded)")
print(f"✅ Index Name:  {index_name}")

## Step 4: Create Search Index Schema

We define a simple index with 4 fields:
- `id` — unique document key
- `title` — section heading (searchable, filterable)
- `content` — full section text (searchable)
- `category` — auto-assigned category (filterable, facetable)

In [ ]:
index_client = SearchIndexClient(
    endpoint=search_endpoint, credential=credential
)

fields = [
    SimpleField(
        name="id", type=SearchFieldDataType.String,
        key=True, filterable=True,
    ),
    SearchableField(
        name="title", type=SearchFieldDataType.String,
        filterable=True, sortable=True,
    ),
    SearchableField(
        name="content", type=SearchFieldDataType.String,
    ),
    SearchableField(
        name="category", type=SearchFieldDataType.String,
        filterable=True, facetable=True,
    ),
]

index = SearchIndex(name=index_name, fields=fields)
result = index_client.create_or_update_index(index)
print(f"✅ Index '{result.name}' created/updated")
print(f"   Fields: {[f.name for f in result.fields]}")

## Step 5: Chunk & Upload Policy Document

We split `contoso_insurance_policy.md` by `##` headings into meaningful sections, auto-categorize each chunk, and upload them to the search index.

In [ ]:
# Read and chunk the policy document
with open(str(POLICY_DOC), "r", encoding="utf-8") as f:
    policy_text = f.read()

sections = re.split(r"\n(?=## )", policy_text)

documents = []
for section in sections:
    section = section.strip()
    if not section or len(section) < 20:
        continue

    lines = section.split("\n", 1)
    title = lines[0].lstrip("#").strip()
    content = section

    # Auto-categorize
    title_lower = title.lower()
    category = "policy"
    for keyword, cat in [
        ("claim", "claims"), ("fraud", "fraud"),
        ("exclu", "exclusions"), ("deduct", "deductibles"),
        ("coverage", "coverage"), ("renewal", "renewal"),
        ("cancel", "renewal"), ("contact", "contact"),
    ]:
        if keyword in title_lower:
            category = cat
            break

    doc_id = hashlib.md5(title.encode()).hexdigest()[:16]
    documents.append({
        "id": doc_id,
        "title": title,
        "content": content,
        "category": category,
    })

print(f"✅ Chunked policy into {len(documents)} sections:")
for doc in documents:
    print(f"   - [{doc['category']:12s}] {doc['title'][:60]}")

In [ ]:
# Upload documents to the index
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=credential,
)

upload_result = search_client.upload_documents(documents=documents)
succeeded = sum(1 for r in upload_result if r.succeeded)
print(f"✅ Uploaded {succeeded}/{len(documents)} documents to '{index_name}'")

## Step 6: Create Search-Grounded Agent

Now we create an agent that uses `AzureAISearchTool` to query our index. The agent will automatically search the index when answering questions.

In [ ]:
search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[
            AISearchIndexResource(
                project_connection_id=connection_id,
                index_name=index_name,
                query_type=AzureAISearchQueryType.SIMPLE,
            ),
        ]
    )
)

agent = project_client.agents.create_version(
    agent_name="smartclaims-search",
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are SmartClaims Knowledge Agent. You have access to "
            "the company's enterprise knowledge base via Azure AI Search. "
            "Use it to answer questions about insurance policies, claims "
            "procedures, precedents, and guidelines. Always cite the "
            "source documents when providing information. If you cannot "
            "find relevant information in the knowledge base, say so."
        ),
        tools=[search_tool],
    ),
)
print(f"✅ Agent: {agent.name} v{agent.version}")
print(f"   Index: {index_name}")
print(f"   Query: SIMPLE (keyword search)")

## Step 7: Enterprise Knowledge Queries

Testing with three types of enterprise knowledge questions:
1. **Claims Procedures** — process and documentation queries
2. **Policy Coverage** — detailed coverage information
3. **Fraud Policy** — detection mechanisms and consequences

In [ ]:
queries = [
    ("Claims Procedures",
     "What is the standard procedure for filing an insurance claim? "
     "What documentation is required?"),
    ("Policy Coverage",
     "What types of coverage are available under the comprehensive "
     "insurance policy? List all covered scenarios."),
    ("Fraud Policy",
     "What are the fraud detection mechanisms and consequences "
     "for fraudulent claims?"),
]

for title, question in queries:
    print(f"\n{'─'*60}")
    print(f"📋 {title}")
    print(f"❓ {question[:80]}")

    response = openai_client.responses.create(
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "version": agent.version,
                "type": "agent_reference",
            }
        },
        input=question,
    )

    answer = response.output_text
    if len(answer) > 600:
        answer = answer[:600] + "..."
    print(f"🤖 {answer}")

## Step 8: FileSearch vs Azure AI Search — Detailed Comparison

| Feature | Lab 3 (FileSearch) | Lab 10 (AI Search) |
|---------|-------------------|-------------------|
| Document scale | Small (< 100 files) | Large (millions of docs) |
| Search type | Vector only | Hybrid (vector + keyword) |
| Index management | Managed by platform | You control the index |
| Security trimming | Not supported | Supported (per-doc ACLs) |
| Custom analyzers | No | Yes (language, phonetic) |
| Existing index | Must re-upload | Connect to existing |
| Scoring profiles | No | Yes (boost/penalize) |
| Semantic ranking | No | Yes (with semantic config) |
| Setup effort | Minimal (just upload) | Moderate (index + connect) |
| Best for | Prototyping, demos | Production, enterprise |

### Decision Guide
- **Start with Lab 3 (FileSearch)** for prototyping and demos
- **Move to Lab 10 (AI Search)** when you need scale, security, or existing indexes

## Step 9: Clean Up

In [ ]:
# Delete agent
project_client.agents.delete(agent.name)
print("✅ Agent deleted")

# Delete the index (comment out to keep it for exploration)
index_client.delete_index(index_name)
print(f"✅ Index '{index_name}' deleted")

## ✅ Lab 10 Complete!

### Key Takeaways
| Concept | What You Learned |
|---------|------------------|
| **Index Creation** | Programmatically create and populate a search index |
| **Document Chunking** | Split Markdown by headings with auto-categorization |
| **Foundry Connections** | Retrieve endpoint + API key from project connections |
| **AzureAISearchTool** | Connect agent to search indexes via connection ID |
| **Enterprise RAG** | Production-grade with security trimming, hybrid search |

---
**Next →** [Lab 11: FastAPI Web App](../labs/lab11_fastapi_webapp.py) — Deploy the complete SmartClaims web application